Monte Carlo Simulation to Computed Adsorption in a Metal-Organic Framework

# Exercise 2: CBMC




In [ ]:
import sys
from raspalib import *
from tqdm.autonotebook import tqdm
from time import sleep

# change input fugacity/pressure [Pa]
inputFugacity = 1e3

# change input temperature [K]
temperature = 200

swapProbability = 0.0
swapCBMCProbability = 1.0
swapCFCMCProbability = 0.0
swapCBCFCMCProbability = 0.0
widomProbability = 0.0

numberOfCycles = 10000
numberOfInitializationCycles = 10000
numberOfEquilibrationCycles = 0

printEvery = 100

atomTypes = [
    PseudoAtom(name="Cu1", frameworkType=True, mass=63.5460, charge=1.248, atomicNumber=29),
    PseudoAtom(name="O1", frameworkType=True, mass=15.9994, charge=-0.624, atomicNumber=8),
    PseudoAtom(name="C1", frameworkType=True, mass=12.0107, charge=0.494, atomicNumber=6),
    PseudoAtom(name="C2", frameworkType=True, mass=12.0107, charge=0.13, atomicNumber=6),
    PseudoAtom(name="C3", frameworkType=True, mass=12.0107, charge=-0.156, atomicNumber=6),
    PseudoAtom(name="H1", frameworkType=True, mass=1.00794, charge=0.156, atomicNumber=1),
    PseudoAtom(name="C_co2", frameworkType=False, mass=12.0, charge=0.6512, atomicNumber=6),
    PseudoAtom(name="O_co2", frameworkType=False, mass=15.9994, charge=-0.3256, atomicNumber=8)
]


parameters = [
    VDWParameters(2.5161, 3.11369),
    VDWParameters(48.1581, 3.03315),
    VDWParameters(47.8562, 3.47299),
    VDWParameters(47.8562, 3.47299),
    VDWParameters(47.8562, 3.47299),
    VDWParameters(7.64893, 2.84642),
    VDWParameters(29.933, 2.745),
    VDWParameters(85.671, 3.017)
]

force_field = ForceField(
    pseudoAtoms=atomTypes,
    parameters=parameters,
    mixingRule=ForceField.MixingRule.Lorentz_Berthelot,
    cutOffFrameworkVDW=12.0,
    cutOffMoleculeVDW=12.0,
    cutOffCoulomb=12.0,
    shifted=True,
    tailCorrections=False,
    useCharge=True,
)


framework = Framework(
    frameworkId=0,
    forceField=force_field,
    componentName="Cu-BTC",
    simulationBox=SimulationBox(26.343, 26.343, 26.343),
    spaceGroupHallNumber=523,
    definedAtoms=[
        Atom(double3(0.2853, 0.2853, 0), 1.248, 1.0, 0, 0, 0, 0),
        Atom(double3(0.3166, 0.2431, 0.9478), -0.624, 1.0, 0, 1, 0, 0),
        Atom(double3(0.2968, 0.2032, 0.9313), 0.494, 1.0, 0, 2, 0, 0),
        Atom(double3(0.322, 0.178, 0.887), 0.130, 1.0, 0, 3, 0, 0),
        Atom(double3(0.3655, 0.1994, 0.8655), -0.156, 1.0, 0, 4, 0, 0),
        Atom(double3(0.3802, 0.228, 0.8802), 0.156, 1.0, 0, 5, 0, 0),
    ],
    numberOfUnitCells=int3(1, 1, 1),
)

move_probabilities = MCMoveProbabilities(
    translationProbability=0.5,
    rotationProbability=0.5,
    reinsertionCBMCProbability=0.5,
    swapProbability=swapProbability,
    swapCBMCProbability=swapCBMCProbability,
    swapCFCMCProbability=swapCFCMCProbability,
    swapCBCFCMCProbability=swapCBCFCMCProbability,
    widomProbability=widomProbability,
)

component = Component(
    componentId=0,
    forceField=force_field,
    componentName="CO2",
    criticalTemperature=304.1282,
    criticalPressure=7377300.0,
    acentricFactor=0.22394,
    definedAtoms=[
        Atom(double3(0.0, 0.0, 1.149), -0.3256, 1.0, 0, 7, 0, 0),
        Atom(double3(0.0, 0.0, 0.0), 0.6512, 1.0, 0, 6, 0, 0),
        Atom(double3(0.0, 0.0, -1.149), -0.3256, 1.0, 0, 7, 0, 0),
    ],
    numberOfBlocks=5,
    numberOfLambdaBins=21,
    particleProbabilities=move_probabilities,
    fugacityCoefficient=1.0,
    thermodynamicIntegration=False,
)

system_probabilities = MCMoveProbabilities()

system = System(
    systemId=0,
    forceField=force_field,
    simulationBox=None, 
    externalTemperature=temperature,
    externalPressure=inputFugacity,
    heliumVoidFraction=0.774, 
    frameworkComponents=framework,
    components=[component],
    initialNumberOfMolecules=[0],
    numberOfBlocks=5,
    systemProbabilities=system_probabilities,
    sampleMoviesEvery=None
)

mc = MonteCarlo(
    numberOfCycles=numberOfCycles,
    numberOfInitializationCycles=numberOfInitializationCycles,
    numberOfEquilibrationCycles=numberOfEquilibrationCycles,
    printEvery=printEvery,
    writeBinaryRestartEvery=5000,
    rescaleWangLandauEvery=2000,
    optimizeMCMovesEvery=2000,
    systems=[system],
    numberOfBlocks=5
)



def progress_call_back_initialization():
    progress_initialization.update(printEvery)
    total_progress.update(printEvery)
    sys.stdout.flush()

def progress_call_back_equilibration():
    progress_equilibration.update(printEvery)
    total_progress.update(printEvery)
    sys.stdout.flush()

def progress_call_back_production():
    progress_production.update(printEvery)
    total_progress.update(printEvery)
    sys.stdout.flush()

numberOfTotalCycles = numberOfCycles + numberOfInitializationCycles + numberOfEquilibrationCycles

with tqdm(total= numberOfTotalCycles, desc="Total", colour='black', position=3) as total_progress:

  progress_initialization = tqdm(total=numberOfInitializationCycles, desc="Initialization", colour='red', position=0)
  progress_equilibration = tqdm(total=numberOfEquilibrationCycles, desc=" Equilibration", colour='magenta', position=1)
  progress_production = tqdm(total=numberOfCycles, desc="    Production", colour='green', position=2)

  # Monte Carlo initialization step
  mc.initialize(call_back_function=progress_call_back_initialization)
     
  # Monte Carlo equilibration step (measuring bias-factors)
  mc.equilibrate(call_back_function=progress_call_back_equilibration)
  
  # Monte Carlo production step
  mc.production(call_back_function=progress_call_back_production)

  progress_initialization.close()
  progress_equilibration.close()
  progress_production.close()


In [ ]:
pressure = mc.systems[0].inputPressure
print(f"The fugacity is: {pressure} Pa\n")
print(mc.systems[0].averageLoadings.writeAveragesStatistics(mc.systems[0].components, mc.systems[0].frameworkMass(), int3(1, 1, 1)))

print(mc.systems[0].writeMCMoveStatistics())

Question 1: Select a low fugacity isotherm point and a high fugacity isotherm point, and check whether using biasing improves the insertion/deletion efficiency. Copy the loading-data  and insertion acceptance ratios into the python-arrays below to plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fugacity_conventional = [1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10]
loading_conventional = [
    1.230550e-01,
    1.158265e00,
    1.002618e01,
    3.424474e01,
    1.819090e02,
    2.011196e02,
    2.140322e02,
    2.196341e02,
    2.247062e02,
    2.258271e02,
    2.255468e02,
]
loading_error_conventional = [
    7.904761e-03,
    7.209103e-02,
    4.833829e-01,
    9.110376e-01,
    2.468011e00,
    1.482671e00,
    1.385070e00,
    7.646734e-01,
    4.057684e-01,
    4.800259e-01,
    7.901519e-01,
]
insertion_acceptance_conventional = [
    0.010035,
    0.021153,
    0.024605,
    0.042101,
    0.000289,
    0.000016,
    0.000001,
    0.000001,
    0.0,
    0.0,
    0.0,
]

# start refactor
fugacity_cbmc = []
loading_cbmc = []
loading_error_cbmc = []
insertion_acceptance_cbmc = []
# end refactor

fig, ax1 = plt.subplots()
ax1.set_title("CBMC vs Conventional insertion scheme 200K")
ax1.set_xlabel("fugacity f / Pa", fontsize=20)
ax1.set_ylabel("Absolute loading / molecules.cell$^{-1}$", fontsize=16)
ax1.set_xscale("log")
ax1.errorbar(
    fugacity_conventional,
    loading_conventional,
    yerr=loading_error_conventional,
    color="purple",
    label="Loading conv.",
    marker="s",
)
ax1.errorbar(fugacity_cbmc, loading_cbmc, yerr=loading_error_cbmc, color="red", label="Loading cbmc", marker="o")
ax1.legend(loc="upper right")
plt.ylim([0, 300])
ax2 = ax1.twinx()
ax2.set_xlabel("fugacity f / Pa", fontsize=20)
ax2.set_ylabel(ylabel="Insertion Acceptance / -", fontsize=16)
ax2.errorbar(
    fugacity_conventional, insertion_acceptance_conventional, color="green", label="acceptance conv.", marker="+"
)
ax2.errorbar(fugacity_cbmc, insertion_acceptance_cbmc, color="blue", label="acceptance cbmc", marker="*")
ax2.legend(loc="upper left")
ax2.set_yscale("log")
plt.ylim([1e-7, 10])

Question 2: Why is the ideal gas Rosenbluth reference weight $\left\langle W^{\text{IG}}\right\rangle$ unity for CO2?

Question 3: The acceptance of insertion/deletions increases with the number of trials orientations. Would it be beneficial to increase the number of trials orientation to 100?

Question 4: What are downsides of the CBMC method?

Question 5: Using Widom particle insertion at 200 Kelvin for imposed fugacities from $10^2$ to $10^{10}$ Pa, the following results are obtained:

|fugacity / Pa | $\mu^\text{exc.}$ / kJ/mol | $\mu^\text{ig}$ / kJ/mol | $\mu$ / kJ/mol | f / Pa|
|---|---|---|---|---|
|1e2  |  -9.204 | -19.276 |  -28.480  | $1.007\times 10^2\pm 0.008\times10^2$}
|1e4  |  -9.636 | -11.230 |  -20.866  | $0.981\times10^4 \pm 0.012\times10^4$|
|1e6  |  -6.062 |  -7.145 |  -13.208  | $0.981\times10^6 \pm 0.097\times10^6$|
|1e8  |  10.662 |  -6.977 |    3.684  | $253.154\times10^8\pm  13931\times10^8$|
|1e10 |  21.277 |  -6.868 |   14.409  | $1.6\times10^{13}\pm 2.5\times 10^{21}$|

When does the Widom particle insertion fail? Is that due to the ideal-gas chemical potential part or due to the excess chemical potential?